# Speaker ID Mariano — Enrolamiento de perfiles de voz

Entrena perfiles de voz tipo Alexa para tu asistente Mariano.

**Flujo:**
1. Grabar muestras en tu **Mac** con `record_samples.py` (micro local, archivos diminutos)
2. Subir la carpeta `samples/` a Google Drive (pocos MB)
3. Descargar modelo WeSpeaker ONNX (~25 MB) en disco efímero de Colab
4. Importar muestras desde Drive → embeddings + centroides
5. Calibrar umbral y exportar `speaker_profiles.json`

**Uso de disco (importante si tienes poco espacio en Drive):**

| Ubicación | Qué guarda | Tamaño típico |
|-----------|-----------|---------------|
| `/content/` (Colab) | Modelo + muestras + trabajo | ~50–150 MB (se borra al cerrar sesión) |
| Google Drive | Solo export final (opcional) | ~30 MB |
| Mac local | Problema original resuelto | 0 MB durante entrenamiento |

Con **7 GB libres en Drive sobra de sobra**. Esto NO es como el wake word Mariano (~25 GB de datasets).

**Requisitos:** muestras grabadas en tu Mac (ver celda 5a) y sincronizadas a Google Drive.

Abrir con kernel **Google Colab** desde Cursor (extensión Colab) o en [colab.research.google.com](https://colab.research.google.com).

## 1. Setup

In [2]:
!pip install -q sherpa-onnx soundfile numpy scipy ipywidgets

from pathlib import Path
import json
import shutil
import zipfile
from datetime import datetime, timezone

import numpy as np
import soundfile as sf
import sherpa_onnx
from IPython.display import display, Markdown, clear_output
import ipywidgets as widgets

# --- Configuración de almacenamiento ---
# Trabajo pesado en disco efímero de Colab (NO consume tu Drive ni tu Mac)
WORK_DIR = Path('/content/wayne-speaker-id')
MODELS_DIR = WORK_DIR / 'models'
SAMPLES_DIR = WORK_DIR / 'samples'
EXPORT_DIR = WORK_DIR / 'export'

# Copia del export final a Google Drive (recomendado — files.download() no funciona desde Cursor)
SAVE_EXPORT_TO_DRIVE = True
DRIVE_EXPORT_DIR = Path('/content/drive/MyDrive/wayne-speaker-id/export')

for directory in (MODELS_DIR, SAMPLES_DIR, EXPORT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

MODEL_NAME = 'wespeaker_en_voxceleb_resnet34.onnx'
MODEL_URL = (
    'https://github.com/k2-fsa/sherpa-onnx/releases/download/'
    'speaker-recongition-models/wespeaker_en_voxceleb_resnet34.onnx'
)
MODEL_PATH = MODELS_DIR / MODEL_NAME

if SAVE_EXPORT_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    print(f'Drive export: {DRIVE_EXPORT_DIR}')
else:
    print('Drive NO montado — ahorras espacio. El ZIP se descarga directo al Mac en la celda 9.')

print(f'Work dir (Colab efímero): {WORK_DIR}')

Drive NO montado — ahorras espacio. El ZIP se descarga directo al Mac en la celda 9.
Work dir (Colab efímero): /content/wayne-speaker-id


## 2. Descargar modelo de embeddings

In [3]:
if not MODEL_PATH.is_file():
    !wget -q -O "{MODEL_PATH}" "{MODEL_URL}"
    print(f'Downloaded {MODEL_NAME}')
else:
    print(f'Model already exists: {MODEL_PATH}')

extractor_config = sherpa_onnx.SpeakerEmbeddingExtractorConfig(
    model=str(MODEL_PATH),
    num_threads=2,
    debug=False,
    provider='cpu',
)
assert extractor_config.validate(), 'Invalid sherpa-onnx config'
extractor = sherpa_onnx.SpeakerEmbeddingExtractor(extractor_config)
EMBEDDING_DIM = extractor.dim
print(f'Embedding dimension: {EMBEDDING_DIM}')

Downloaded wespeaker_en_voxceleb_resnet34.onnx
Embedding dimension: 256


## 3. Configurar personas a enrolar

Edita la lista con los nombres de tu casa (minúsculas, sin espacios).

In [4]:
SPEAKERS = ['miguel']  # Añade más: ['miguel', 'ana', 'carlos']

for speaker in SPEAKERS:
    (SAMPLES_DIR / speaker).mkdir(parents=True, exist_ok=True)

display(Markdown('**Personas configuradas:** ' + ', '.join(SPEAKERS)))

**Personas configuradas:** miguel

## 4. Utilidades de audio y embeddings

In [5]:
def load_audio(path: Path) -> tuple[np.ndarray, int]:
    data, sample_rate = sf.read(path, always_2d=True, dtype='float32')
    return np.ascontiguousarray(data[:, 0]), sample_rate


def l2_normalize(embedding: np.ndarray) -> np.ndarray:
    norm = np.linalg.norm(embedding)
    return embedding if norm == 0 else embedding / norm


def compute_embedding(wav_path: Path) -> np.ndarray:
    samples, sample_rate = load_audio(wav_path)
    stream = extractor.create_stream()
    stream.accept_waveform(sample_rate=sample_rate, waveform=samples)
    stream.input_finished()
    assert extractor.is_ready(stream), f'Extractor not ready: {wav_path}'
    return l2_normalize(np.asarray(extractor.compute(stream), dtype=np.float32))


def compute_centroid(wav_files: list[Path]) -> np.ndarray:
    embeddings = [compute_embedding(path) for path in wav_files]
    return l2_normalize(np.mean(np.stack(embeddings, axis=0), axis=0))


def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(l2_normalize(a), l2_normalize(b)))


def list_wavs(speaker: str) -> list[Path]:
    folder = SAMPLES_DIR / speaker
    return sorted(folder.glob('*.wav'))

print('Utilities loaded.')

Utilities loaded.


## 5a. Importar muestras grabadas en tu Mac (recomendado)

El micrófono se graba **en local** (el kernel de Colab es remoto y no accede a tu micro; la grabación por navegador no funciona desde Cursor).

**En tu Mac (terminal):**
```bash
python3 infrastructure/voice/speaker-id/record_samples.py --speaker miguel --count 10
# repite para cada persona: --speaker ana, etc.
```

**Luego sube la carpeta `samples/` a Google Drive** (`MyDrive/wayne-speaker-id/samples/`) — son pocos MB, caben de sobra en 7 GB. Arrástrala en [drive.google.com](https://drive.google.com) o usa Google Drive para escritorio.

La siguiente celda monta Drive e importa las muestras al entorno de Colab.

In [9]:
# Importa las muestras grabadas en tu Mac desde Google Drive.
# Estructura esperada en Drive:
#   MyDrive/wayne-speaker-id/samples/miguel/clip_01.wav
#   MyDrive/wayne-speaker-id/samples/ana/clip_01.wav
from google.colab import drive

drive.mount('/content/drive')

DRIVE_SAMPLES_DIR = Path('/content/drive/MyDrive/wayne-speaker-id/samples')

if DRIVE_SAMPLES_DIR.is_dir():
    imported = 0
    for person_dir in sorted(DRIVE_SAMPLES_DIR.iterdir()):
        if not person_dir.is_dir():
            continue
        dest = SAMPLES_DIR / person_dir.name
        dest.mkdir(parents=True, exist_ok=True)
        for wav in person_dir.glob('*.wav'):
            shutil.copy2(wav, dest / wav.name)
            imported += 1
    print(f'Importados {imported} clip(s) desde {DRIVE_SAMPLES_DIR}')
else:
    print(f'No existe {DRIVE_SAMPLES_DIR}')
    print('Crea la carpeta en Drive y sube los WAV grabados con record_samples.py')

for speaker in SPEAKERS:
    print(f'{speaker}: {len(list_wavs(speaker))} clip(s)')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Importados 10 clip(s) desde /content/drive/MyDrive/wayne-speaker-id/samples
miguel: 10 clip(s)


## 5b. (Alternativa) Subir ZIP — solo en la UI web de Colab

Usa esto **solo si abres el notebook en [colab.research.google.com](https://colab.research.google.com)** (el widget de subida no funciona desde Cursor). Útil para clips capturados con el Satellite1 desde `/share/assist_pipeline`.

1. Organiza en ZIP: `miguel/clip01.wav`, `ana/clip01.wav`, ...
2. Sube aquí

Si usas Drive (celda 5a), **omite esta celda**.

In [8]:
from google.colab import files

uploaded = files.upload()

for filename, content in uploaded.items():
    dest = SAMPLES_DIR / filename
    dest.write_bytes(content)
    print(f'Uploaded: {dest}')

    if filename.lower().endswith('.zip'):
        with zipfile.ZipFile(dest, 'r') as archive:
            archive.extractall(SAMPLES_DIR)
        print(f'Extracted ZIP to {SAMPLES_DIR}')
        dest.unlink()

for speaker in SPEAKERS:
    wavs = list_wavs(speaker)
    print(f'{speaker}: {len(wavs)} clip(s)')

KeyboardInterrupt: 

## 6. Calcular centroides por persona

In [10]:
centroids: dict[str, np.ndarray] = {}
clip_counts: dict[str, int] = {}

for speaker in SPEAKERS:
    wavs = list_wavs(speaker)
    if len(wavs) < 3:
        print(f'WARNING: {speaker} has only {len(wavs)} clips (recommend 8+)')
    if not wavs:
        raise ValueError(f'No WAV files for speaker: {speaker}')

    centroids[speaker] = compute_centroid(wavs)
    clip_counts[speaker] = len(wavs)
    print(f'{speaker}: centroid from {len(wavs)} clip(s)')

display(Markdown('**Centroides calculados.**'))

miguel: centroid from 10 clip(s)


**Centroides calculados.**

## 7. Calibrar umbral (EER aproximado)

In [11]:
def evaluate_pairs() -> tuple[list[float], list[float]]:
    genuine: list[float] = []
    impostor: list[float] = []

    for speaker in SPEAKERS:
        centroid = centroids[speaker]
        for wav_path in list_wavs(speaker):
            embedding = compute_embedding(wav_path)
            genuine.append(cosine_similarity(embedding, centroid))

        for other, other_centroid in centroids.items():
            if other == speaker:
                continue
            for wav_path in list_wavs(speaker):
                embedding = compute_embedding(wav_path)
                impostor.append(cosine_similarity(embedding, other_centroid))

    return genuine, impostor


def estimate_threshold(genuine: list[float], impostor: list[float]) -> float:
    """Calibrate threshold; handles single-speaker (no impostor pairs)."""
    DEFAULT_THRESHOLD = 0.6

    if impostor:
        thresholds = np.linspace(0.0, 1.0, 200)
        best_threshold = DEFAULT_THRESHOLD
        best_distance = 1.0
        for threshold in thresholds:
            far = sum(score >= threshold for score in impostor) / len(impostor)
            frr = sum(score < threshold for score in genuine) / len(genuine)
            distance = abs(far - frr)
            if distance < best_distance:
                best_distance = distance
                best_threshold = float(threshold)
        return max(best_threshold, DEFAULT_THRESHOLD)

    # Single speaker: floor from genuine distribution (percentile 5 - margin)
    if not genuine:
        return DEFAULT_THRESHOLD
    floor = max(DEFAULT_THRESHOLD, float(np.percentile(genuine, 5)) - 0.05)
    return round(min(floor, 0.85), 4)


genuine_scores, impostor_scores = evaluate_pairs()
THRESHOLD = estimate_threshold(genuine_scores, impostor_scores)
MARGIN = 0.05

print(f'Genuine scores: min={min(genuine_scores):.3f}, max={max(genuine_scores):.3f}, mean={np.mean(genuine_scores):.3f}')
if impostor_scores:
    print(f'Impostor scores: min={min(impostor_scores):.3f}, max={max(impostor_scores):.3f}, mean={np.mean(impostor_scores):.3f}')
else:
    print('Single speaker — threshold from genuine score distribution (not EER)')
print(f'Calibrated threshold: {THRESHOLD:.3f}')
print(f'Margin: {MARGIN}')

Genuine scores: min=0.773, max=0.914, mean=0.864
Calibrated threshold (EER approx): 0.000
Margin: 0.05


## 8. Evaluación — matriz de confusión

In [12]:
def identify(embedding: np.ndarray) -> tuple[str, float]:
    scores = {name: cosine_similarity(embedding, centroid) for name, centroid in centroids.items()}
    ranked = sorted(scores.items(), key=lambda item: item[1], reverse=True)
    best_name, best_score = ranked[0]
    second_score = ranked[1][1] if len(ranked) > 1 else 0.0

    if best_score < THRESHOLD or (best_score - second_score) < MARGIN:
        return 'desconocido', best_score
    return best_name, best_score


confusion: dict[str, dict[str, int]] = {speaker: {s: 0 for s in SPEAKERS + ['desconocido']} for speaker in SPEAKERS}

for speaker in SPEAKERS:
    for wav_path in list_wavs(speaker):
        predicted, _ = identify(compute_embedding(wav_path))
        confusion[speaker][predicted] += 1

print('Confusion matrix (rows=true, cols=predicted):')
header = ['true\\pred'] + SPEAKERS + ['desconocido']
print('\t'.join(header))
for speaker in SPEAKERS:
    row = [speaker] + [str(confusion[speaker][col]) for col in SPEAKERS + ['desconocido']]
    print('\t'.join(row))

correct = sum(confusion[s][s] for s in SPEAKERS)
total = sum(sum(confusion[s].values()) for s in SPEAKERS)
print(f'\nAccuracy: {correct}/{total} = {100 * correct / max(total, 1):.1f}%')

Confusion matrix (rows=true, cols=predicted):
true\pred	miguel	desconocido
miguel	10	0

Accuracy: 10/10 = 100.0%


## 9. Exportar artefactos

Genera `speaker_profiles.json`, copia el modelo ONNX y crea ZIP para copiar a HAOS `/share/speaker-id/`.

In [13]:
profiles = {
    'version': 1,
    'model': MODEL_NAME,
    'embedding_dim': EMBEDDING_DIM,
    'threshold': round(THRESHOLD, 4),
    'margin': MARGIN,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'speakers': {
        name: {
            'centroid': centroid.tolist(),
            'clip_count': clip_counts[name],
        }
        for name, centroid in centroids.items()
    },
}

config = {
    'model': MODEL_NAME,
    'threshold': round(THRESHOLD, 4),
    'margin': MARGIN,
    'speakers': SPEAKERS,
}

profiles_path = EXPORT_DIR / 'speaker_profiles.json'
config_path = EXPORT_DIR / 'config.json'
model_export = EXPORT_DIR / MODEL_NAME

profiles_path.write_text(json.dumps(profiles, indent=2), encoding='utf-8')
config_path.write_text(json.dumps(config, indent=2), encoding='utf-8')
shutil.copy2(MODEL_PATH, model_export)

zip_path = EXPORT_DIR / 'speaker-id-mariano-export.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    archive.write(profiles_path, profiles_path.name)
    archive.write(config_path, config_path.name)
    archive.write(model_export, model_export.name)

zip_size_mb = zip_path.stat().st_size / (1024 * 1024)

# --- Guardar en Google Drive (recomendado — files.download() no funciona desde Cursor) ---
drive_zip = None
drive_mounted = Path('/content/drive/MyDrive').is_dir()
if drive_mounted:
    DRIVE_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    drive_zip = DRIVE_EXPORT_DIR / zip_path.name
    shutil.copy2(zip_path, drive_zip)
    shutil.copy2(profiles_path, DRIVE_EXPORT_DIR / profiles_path.name)
    shutil.copy2(config_path, DRIVE_EXPORT_DIR / config_path.name)
    shutil.copy2(model_export, DRIVE_EXPORT_DIR / model_export.name)
    print(f'Guardado en Drive: {DRIVE_EXPORT_DIR}')
    for f in sorted(DRIVE_EXPORT_DIR.iterdir()):
        print(f'  {f.name} ({f.stat().st_size / 1024:.0f} KB)')
else:
    print('AVISO: Drive no montado. Ejecuta la celda 5a primero, o usa train_profiles.py en el Mac.')

display(Markdown(f"""
### Export complete (~{zip_size_mb:.1f} MB)

| Ubicación | Ruta |
|-----------|------|
| Colab (efímero) | `{EXPORT_DIR}` |
| Google Drive | `{drive_zip or 'NO — monta Drive en celda 5a'}` |

**Siguiente paso:** abre Drive → `MyDrive/wayne-speaker-id/export/` y descarga el ZIP.
Luego extráelo en HAOS `/share/speaker-id/` vía Samba.
"""))

# files.download() solo funciona en la UI web de Colab, no desde Cursor — omitido


### Export complete (~23.5 MB)

| File | Path |
|------|------|
| Profiles | `/content/wayne-speaker-id/export/speaker_profiles.json` |
| Model | `/content/wayne-speaker-id/export/wespeaker_en_voxceleb_resnet34.onnx` |
| ZIP | `/content/wayne-speaker-id/export/speaker-id-mariano-export.zip` |


**Next steps:**
1. Descarga el ZIP al Mac (abajo) — no necesitas espacio en Drive
2. Extrae en HAOS `/share/speaker-id/` vía Samba
3. Instala add-on `speaker-id-mariano` y arráncalo
4. Di un comando al Satellite1 → comprueba `input_text.current_speaker`


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded speaker-id-mariano-export.zip (23.5 MB)


## 10. Validar export (opcional)

In [14]:
# Inline validation (same logic as infrastructure/voice/speaker-id/validate_profiles.py)
required_keys = {'version', 'model', 'embedding_dim', 'threshold', 'margin', 'speakers'}
missing = required_keys - set(profiles.keys())
assert not missing, f'Missing keys: {missing}'
assert profiles['speakers'], 'No speakers enrolled'
for name, data in profiles['speakers'].items():
    assert len(data['centroid']) == profiles['embedding_dim'], f'{name}: dim mismatch'
print('Validation OK — ready to deploy to HAOS')

Validation OK — ready to deploy to HAOS
